In [1]:
# ============================================================
# Notebook : 00_data_preparation.ipynb
# Purpose  : Step 1 (Sample Eligibility) and Step 2 (Label Validity)
#            of the sensor-based underwriting verification framework.
#            Parse the integrated judgement reason into five abnormal
#            categories, redefine the analysis unit from household to
#            observation session, separate the structurally distinct
#            facility track, apply a minimum-length criterion, and
#            record the sample-coverage and label-decomposition tables.
# Design   : All code lives in a single cell by convention.
#            No absolute path is printed (anonymity for double-blind review).
# ============================================================

import re
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

# ---- Relative paths (resolved like a Jupyter working dir; never printed) ----
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA = ROOT / "data"
TAB = ROOT / "results" / "tables"
TAB.mkdir(parents=True, exist_ok=True)

# ---- Load raw observation-level records (10-minute cadence) ----
df = pd.read_pickle(DATA / "raw_sessions.pkl")
df = df.sort_values(["HH_ID", "Timestamp"]).reset_index(drop=True)
# SkinTemperature is recorded as a skin-temperature *change* value; it is the
# channel that defines the Skin-temp (ST) abnormal category and is central to
# the intervention-bypass analysis of Steps 4-5.

# ============================================================
# Step 2. Label validity: decompose the integrated reason text
#         into five mutually interpretable abnormal categories.
#         A regular-expression parse of the free-text reason is
#         used instead of the single integrated label, so that
#         each component carries its own occurrence mechanism.
# ============================================================
REASON_KEYWORDS = {
    "HR": "분당 심박",        # heart-rate abnormality
    "BR": "분당 호흡",        # breath-rate abnormality
    "ST": "피부온도변화",      # skin-temperature change (no dedicated sensor channel)
    "SP": "혈중산소농도",      # blood-oxygen (SpO2) abnormality
    "ER_btn": "응급버튼",     # emergency button (direct signal)
    "ER_voice": "응급음성",    # emergency voice (direct signal)
}
reason = df["Reason"].fillna("")
for code, kw in REASON_KEYWORDS.items():
    df["cat_" + code] = reason.str.contains(kw, regex=False).astype(int)

# Merge the two direct emergency signals into one category.
df["cat_ER"] = ((df["cat_ER_btn"] + df["cat_ER_voice"]) > 0).astype(int)
df = df.drop(columns=["cat_ER_btn", "cat_ER_voice"])

ABN = ["HR", "BR", "ST", "SP", "ER"]
CAT_COLS = ["cat_" + c for c in ABN]

# Severity carried by the same reason text.
df["sev_danger"] = reason.str.contains("위험", regex=False).astype(int)
df["sev_caution"] = reason.str.contains("주의", regex=False).astype(int)
df["any_abnormal"] = (df[CAT_COLS].sum(axis=1) > 0).astype(int)

# Parse coverage: share of non-"일상" records whose reason was parsed into
# at least one category (records that merely say "특이사항 없음" are normal).
non_normal = ~reason.str.contains("특이사항 없음", regex=False)
parse_coverage = df.loc[non_normal, "any_abnormal"].mean()

# ============================================================
# Step 1. Sample eligibility.
#   (a) Redefine the analysis unit: household -> observation session.
#       A session is a maximal run of consecutive 10-minute records
#       with no internal gap exceeding SESSION_GAP_MIN minutes.
#   (b) Separate the facility track: the two households whose residential
#       environment is classified "other" (Environment == '기타'), i.e. a
#       structurally distinct observation environment (shared/facility)
#       with re-identification risk and a far higher danger rate.
#   (c) Flag sessions long enough for causal discovery (>= MIN_PCMCI_LEN
#       steps = 24 h); the main home dataset keeps all home records.
# ============================================================
SESSION_GAP_MIN = 15      # a gap > 15 min starts a new session
MIN_PCMCI_LEN = 144       # sessions with >= 144 steps (24 h) feed causal discovery
FACILITY_IDS = sorted(df.loc[df["Environment"] == "기타", "HH_ID"].unique().tolist())

df["track"] = np.where(df["Environment"] == "기타", "facility", "home")

# Session segmentation.
gap_min = df.groupby("HH_ID")["Timestamp"].diff().dt.total_seconds().div(60)
new_session = (gap_min > SESSION_GAP_MIN) | (gap_min.isna())
df["session_idx"] = new_session.groupby(df["HH_ID"]).cumsum().astype(int)
df["session_id"] = df["HH_ID"] + "_S" + df["session_idx"].astype(str).str.zfill(4)

# Session length; flag sessions eligible for causal discovery (>= 24 h).
sess_len = df.groupby("session_id")["Timestamp"].transform("size")
df["session_len"] = sess_len
df["pcmci_eligible"] = (df["session_len"] >= MIN_PCMCI_LEN).astype(int)

# ---- Sample-coverage table (household level) ----
cov = (
    df.groupby("HH_ID")
    .agg(
        track=("track", "first"),
        region=("Region", "first"),
        age=("Age", "first"),
        gender=("Gender", "first"),
        n_obs=("Timestamp", "size"),
        n_sessions=("session_id", "nunique"),
        danger_rate=("sev_danger", "mean"),
    )
    .reset_index()
)
kept = (
    df[df["pcmci_eligible"] == 1]
    .groupby("HH_ID")
    .agg(n_sessions_pcmci=("session_id", "nunique"), n_obs_pcmci=("Timestamp", "size"))
    .reset_index()
)
cov = cov.merge(kept, on="HH_ID", how="left").fillna({"n_sessions_pcmci": 0, "n_obs_pcmci": 0})
cov["pcmci_coverage"] = (cov["n_obs_pcmci"] / cov["n_obs"]).round(4)
cov = cov.sort_values(["track", "HH_ID"]).reset_index(drop=True)
cov.to_csv(TAB / "table_sample_coverage.csv", index=False, encoding="utf-8-sig")

# ---- Label-decomposition table (category level, home track; all records) ----
home_obs = df[df["track"] == "home"]
rows = []
for c in ABN:
    col = "cat_" + c
    rows.append(
        {
            "category": c,
            "n_positive": int(home_obs[col].sum()),
            "prevalence": round(home_obs[col].mean(), 5),
            "n_danger": int(((home_obs[col] == 1) & (home_obs["sev_danger"] == 1)).sum()),
            "n_caution": int(((home_obs[col] == 1) & (home_obs["sev_caution"] == 1)).sum()),
        }
    )
label_decomp = pd.DataFrame(rows).sort_values("n_positive", ascending=False).reset_index(drop=True)
label_decomp.to_csv(TAB / "table_label_decomposition.csv", index=False, encoding="utf-8-sig")

# ============================================================
# Persist the analysis-ready datasets.
#   - analysis_home.pkl    : 25 home households, kept sessions (main analysis)
#   - facility_track.pkl   : 2 facility households (qualitative case track)
# ============================================================
# Track A time-series sensor channels used for causal discovery and the
# classifier (environment + vital signs). Sleep / stress metrics belong to the
# Track B functional-status covariates and are kept in the data but not used as
# time-series precursor candidates.
SENSOR_CONT = [
    "Temperature", "Humidity", "Illuminance", "Activity_IR", "CO2", "TVOC",
    "HeartRate", "BreathRate", "SPO2", "SkinTemperature",
]
SENSOR_TRACKB = ["SleepPhase", "SleepScore", "StressIndex"]
SENSOR_RAW_ER = ["Button", "Shout"]
KEEP_COLS = (
    ["HH_ID", "session_id", "session_idx", "Timestamp", "track",
     "Age", "Gender", "Region", "Environment", "DiseaseYN", "Estimation", "Reason"]
    + SENSOR_CONT + SENSOR_TRACKB + SENSOR_RAW_ER + CAT_COLS
    + ["cat_ER", "sev_danger", "sev_caution", "any_abnormal",
       "session_len", "pcmci_eligible"]
)
KEEP_COLS = list(dict.fromkeys(KEEP_COLS))  # de-dup, preserve order

home = df[df["track"] == "home"][KEEP_COLS].reset_index(drop=True)
facility = df[df["track"] == "facility"][KEEP_COLS].reset_index(drop=True)
home.to_pickle(DATA / "analysis_home.pkl")
facility.to_pickle(DATA / "facility_track.pkl")

# Save the variable dictionary for downstream notebooks (single source of truth).
meta = {
    "sensor_cont": SENSOR_CONT,
    "sensor_trackb": SENSOR_TRACKB,
    "sensor_raw_er": SENSOR_RAW_ER,
    "abnormal_categories": ABN,
    "category_defining_sensor": {   # which sensor channel defines each category
        "HR": "HeartRate", "BR": "BreathRate", "SP": "SPO2",
        "ST": "SkinTemperature",    # skin-temperature change channel
        "ER": "Button/Shout",
    },
    "session_gap_min": SESSION_GAP_MIN,
    "min_pcmci_len": MIN_PCMCI_LEN,
    "facility_ids": FACILITY_IDS,
}
pd.to_pickle(meta, DATA / "var_meta.pkl")

# ============================================================
# Console summary (counts and rates only; no file paths).
# ============================================================
print("=== Step 1-2 : data preparation summary ===")
print(f"Raw observation records      : {len(df):,}")
print(f"Households (total)           : {df['HH_ID'].nunique()}")
print(f"  home track                 : {cov[cov.track=='home'].shape[0]}")
print(f"  facility track (Env='기타') : {cov[cov.track=='facility'].shape[0]}  -> {FACILITY_IDS}")
print(f"Home analysis records        : {len(home):,}")
print(f"Facility records             : {len(facility):,}")
print(f"Home sessions (gap>{SESSION_GAP_MIN}min)       : {home['session_id'].nunique():,}")
print(f"Home sessions >= {MIN_PCMCI_LEN} steps (PCMCI): "
      f"{df[(df.track=='home')&(df.pcmci_eligible==1)]['session_id'].nunique():,} "
      f"({df[(df.track=='home')&(df.pcmci_eligible==1)]['HH_ID'].nunique()} households)")
print(f"Reason->category parse coverage (non-normal): {parse_coverage:.4f}")
print()
print("=== Label decomposition (home track, kept sessions) ===")
print(label_decomp.to_string(index=False))
print()
print("=== Facility vs home danger rate ===")
print(cov.groupby('track')['danger_rate'].mean().round(4).to_string())
print()
print("Saved: analysis_home.pkl, facility_track.pkl, var_meta.pkl,")
print("       table_sample_coverage.csv, table_label_decomposition.csv")


=== Step 1-2 : data preparation summary ===
Raw observation records      : 109,819
Households (total)           : 27
  home track                 : 25
  facility track (Env='기타') : 2  -> ['A0029', 'A0030']
Home analysis records        : 99,161
Facility records             : 10,658
Home sessions (gap>15min)       : 534
Home sessions >= 144 steps (PCMCI): 163 (22 households)
Reason->category parse coverage (non-normal): 1.0000

=== Label decomposition (home track, kept sessions) ===
category  n_positive  prevalence  n_danger  n_caution
      HR       14370     0.14492       836      13534
      BR        6956     0.07015       474       6482
      ST        4664     0.04703      1209       3455
      SP         569     0.00574         5        564
      ER         314     0.00317       314          0

=== Facility vs home danger rate ===
track
facility    0.1010
home        0.0242

Saved: analysis_home.pkl, facility_track.pkl, var_meta.pkl,
       table_sample_coverage.csv, table_label_d